# 02 — GPU Memory Hierarchy & Roofline

Concepts: **memory hierarchy** (registers → shared → L2 → global) and **roofline** (memory-bound vs compute-bound). See `docs/gpu_memory_hierarchy.md` and `docs/memory_hierarchy_diagrams.md`.

## Memory hierarchy (RTX 3050)

| Level | Scope | Latency | Use |
|-------|--------|---------|-----|
| Registers | Per-thread | ~0 | Locals, accumulators |
| Shared | Per-block | ~20–30 cycles | Tiles, reductions |
| L1/L2 | Per-SM/GPU | Variable | Cached global access |
| Global | GPU | 200–400 cycles | Inputs/outputs |

## Roofline

- **Arithmetic intensity** = FLOPs / (bytes read + written).
- **Ridge point** = peak_FLOPS / peak_BW. Below ridge → **memory-bound**; above → **compute-bound**.
- RTX 3050 (FP16): ~9 TFLOPS, ~192 GB/s → ridge ≈ 47 FLOP/byte.

In [ ]:
# Example: approximate arithmetic intensity for matmul NxN (FP16)
def ai_matmul(N):
    flops = 2 * N**3
    bytes_rw = 3 * N * N * 2  # A, B, C in FP16
    return flops / bytes_rw

for N in [256, 1024, 4096]:
    print(f"Matmul {N}x{N}: AI = {ai_matmul(N):.1f} FLOP/byte")
print("Ridge (RTX 3050 FP16) ≈ 47 → matmul 1024 is compute-bound.")

In [ ]:
# Generate roofline plot (requires matplotlib)
import sys
from pathlib import Path
sys.path.insert(0, str(Path("..").resolve()))
try:
    from profiling.roofline_analysis.plot_roofline import plot_roofline
    out = plot_roofline(B=8, S=256, H=768)
    print("Saved:", out)
except Exception as e:
    print("Run from repo root: python profiling/roofline_analysis/plot_roofline.py")